# ClimaTwin India — Temperature hi-res diffusion (Colab GPU)

Trains the **CorrDiff-style residual-diffusion downscaler** (0.25° → 0.05°) for **Tmax** and **Tmin**
on real **INDmet** 0.05° truth, on a free Colab GPU. Rainfall is already trained locally; this notebook
fills in the two temperature variables so the `/highres` (diffusion ensemble) panel works for all three.

**Pipeline:** clone code → download+build the INDmet 0.05° cube (pilot box only, small) → train Tmax →
train Tmin → zip the cube + checkpoints + metrics → download to your Mac.

**Honest caveat (keep it in the deck):** temperature fields are smooth, so the diffusion *RMSE* gain over
bilinear is modest. The real win is **spectral/CRPS** skill (sharp, plausible texture) — exactly how
NVIDIA Earth-2 CorrDiff is scored. The notebook prints that honest story per variable.

> Runtime → **Change runtime type → GPU (T4 is fine)**. End-to-end ≈ download ~1–2 h + ~20–40 min/var.

## 1 · Check the GPU

In [ ]:
!nvidia-smi -L

## 2 · Get the code

**Option A (default):** clone the public repo — always matches `origin/main`.

**Option B:** if you'd rather upload the `climatwin_code.zip` I generated, comment out the clone and
uncomment the upload block.

In [ ]:
import os

# --- Option A: clone (default) ---
if not os.path.isdir('climatwin-india'):
    !git clone --depth 1 https://github.com/ayushap18/climatwin-india.git
REPO_DIR = '/content/climatwin-india'

# --- Option B: upload the zip instead (uncomment) ---
# from google.colab import files
# up = files.upload()                       # pick climatwin_code.zip
# import zipfile, io
# name = next(iter(up))
# with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall('/content/code')
# REPO_DIR = '/content/code'                # adjust if the zip nests a folder

os.chdir(REPO_DIR)
print('working dir:', os.getcwd())
!ls

## 3 · Install dependencies

Colab already has torch + numpy + scipy. We add the geo/data bits (`xarray`, `netCDF4`, `remotezip`).

In [ ]:
!pip install -q xarray netCDF4 remotezip

## 4 · Download + build the INDmet 0.05° cube

`data.ingest_indmet` pulls **only** the per-variable/per-year members it needs from the 16 GB Zenodo zip
via HTTP range requests, clips each to the Delhi-NCR box, and writes one small cube
(`data/indmet_cube_005.nc`). Ingesting all three vars keeps the cube consistent with what the backend
serves. The raw download is the slow part (~1–2 h); the clipped cube itself is only a few hundred MB.

Years follow `config.SPLIT` (train ≤2018 / val 2019–21 / test 2022–23) automatically.

In [ ]:
# all three vars so the cube the backend opens has rainfall+tmax+tmin
!python -m data.ingest_indmet --vars rainfall tmax tmin --years 2000 2023

## 5 · Train the **Tmax** diffusion downscaler (~20–40 min)

DDPM (cosine schedule) on the residual = truth − bilinear, conditioned on the bilinear field;
train-years-only norm stats (no leakage). Saves `models/checkpoints/diffusion_downscale_tmax.pt` and
auto-runs the honest TEST-split eval (RMSE + CRPS + hi-wavenumber spectrum vs bilinear).

In [ ]:
!python -m models.diffusion_downscale --var tmax --epochs 120

## 6 · Train the **Tmin** diffusion downscaler (~20–40 min)

In [ ]:
!python -m models.diffusion_downscale --var tmin --epochs 120

## 7 · (Optional) retrain **rainfall** on this same cube

Rainfall is already trained locally (FSS is its headline metric). Only run this if you want all three
trained from one identical cube. Skip otherwise.

In [ ]:
# !python -m models.diffusion_downscale --var rainfall --epochs 120

## 8 · Zip the results + download

Bundles the **cube** (the backend needs it to serve any diffusion var), both **checkpoints**, and the
**metrics** JSONs. Download, then drop them into your local repo (next cell tells you where).

In [ ]:
import zipfile, glob, os
out = '/content/climatwin_diffusion_temp.zip'
wanted = [
    'data/indmet_cube_005.nc',
    'models/checkpoints/diffusion_downscale_tmax.pt',
    'models/checkpoints/diffusion_downscale_tmin.pt',
    'models/diffusion_metrics_tmax.json',
    'models/diffusion_metrics_tmin.json',
]
# include rainfall artifacts too if cell 7 was run
for extra in ['models/checkpoints/diffusion_downscale.pt', 'models/diffusion_metrics.json']:
    if os.path.exists(extra): wanted.append(extra)
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in wanted:
        if os.path.exists(f):
            z.write(f); print('added', f, f'({os.path.getsize(f)/1e6:.1f} MB)')
        else:
            print('MISSING (skipped):', f)
print('\nzip:', out, f'({os.path.getsize(out)/1e6:.1f} MB)')
from google.colab import files
files.download(out)

## ✅ Done — finish on your Mac

Unzip `climatwin_diffusion_temp.zip` **at the repo root** so the files land in place:

```bash
cd ~/'hack isro'
unzip -o ~/Downloads/climatwin_diffusion_temp.zip
# -> data/indmet_cube_005.nc
# -> models/checkpoints/diffusion_downscale_tmax.pt  (+ _tmin)
# -> models/diffusion_metrics_tmax.json              (+ _tmin)
```

Then restart the backend — `/meta.diffusion_vars` will now list `tmax` and `tmin`, and the Downscale
view's **DIFFUSION ENSEMBLE** panel + the `0.05° HI-RES` toggle light up for temperature too:

```bash
make serve            # or your `twin` alias
```

Sanity check: `curl localhost:8000/meta | python -m json.tool | grep -A4 diffusion_vars`.